In [15]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 --driver-memory 4g pyspark-shell"
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("phase6-streaming").getOrCreate()
print("phase6-streaming up")

phase6-streaming up


In [16]:
# ---- Cell 1: persist the serving contract + write the streamable test window (producer input) ----
# Replicates notebook-06 preprocessing deterministically and rules it to disk, exactly like a
# production serving stack would: meds.json (imputations), idx_<col>/ (indexer models), cols.json
# (feature list in training order), source_test.parquet (what the producer replays).
# If data/stream/prep exists, the contract is loaded instead of re-fit (fast restarts).
import os, json, shutil, time as T
from pyspark.ml.feature import StringIndexer

t0 = T.time()
prep, srcp = "/home/jovyan/work/data/stream/prep", "/home/jovyan/work/data/stream/source_test.parquet"

if os.path.isdir(prep) and os.path.isdir(srcp):
    FEATURES = json.load(open(prep + "/cols.json"))
    print(f"serving contract loaded from data/stream/prep (no refit) in {T.time()-t0:.1f}s")
else:
    if os.path.isdir(prep):                    # half-wiped state: start clean, the rule is atomic
        shutil.rmtree(prep)
    tx = spark.read.parquet("/home/jovyan/work/data/processed/train_features")
    max_day = tx.agg(F.max("day")).collect()[0][0]
    cutoff  = max_day - 17
    train, test = tx.filter(F.col("day") <= cutoff), tx.filter(F.col("day") > cutoff)

    num_cols = [c for c, t in train.dtypes if t in ("int", "bigint", "double", "float")
                and c not in ("isFraud", "TransactionID")]
    med_map = {}
    for c, m in zip(num_cols, train.approxQuantile(num_cols, [0.5], 0.001)):
        if not m or m[0] is None:
            continue
        t = next(t for cc, t in train.dtypes if cc == c)
        med_map[c] = int(round(m[0])) if t in ("int", "bigint") else float(m[0])
    train = train.na.fill(med_map)
    test  = test.na.fill(med_map)

    str_cols = [c for c, t in train.dtypes if t == "string"]
    train = train.drop(*[c + "_idx" for c in str_cols])
    test  = test.drop(*[c + "_idx" for c in str_cols])
    train = train.na.fill("__NA__", subset=str_cols)
    test  = test.na.fill("__NA__", subset=str_cols)
    for c in str_cols:
        m = StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep").fit(train)
        m.write().save(prep + "/idx_" + c)
        train = m.transform(train).drop(c)
        test  = m.transform(test).drop(c)

    for c, t in train.dtypes:                   # maxBins armor: idx cols to float, per the nb-06 rule
        if c.endswith("_idx"):
            train = train.withColumn(c, F.col(c) * 1.0)
            test  = test.withColumn(c, F.col(c) * 1.0)

    FEATURES = [c for c, t in train.dtypes if t in ("int", "bigint", "double", "float")
                and c not in ("isFraud", "TransactionID", "TransactionDT", "weight")]
    os.makedirs(prep, exist_ok=True)
    json.dump(FEATURES, open(prep + "/cols.json", "w"))
    json.dump(med_map, open(prep + "/meds.json", "w"))
    test.write.mode("overwrite").parquet(srcp)
    print(f"serving contract computed + saved ({len(FEATURES)} features, {len(str_cols)} indexers) "
          f"in {T.time()-t0:.1f}s")

test_pp = spark.read.parquet(srcp)               # authoritative either way - both paths converge here
from pyspark.ml.classification import GBTClassificationModel
from pyspark.ml.feature import VectorAssembler
model     = GBTClassificationModel.load("/home/jovyan/work/models/gbt_winner")
assembler = VectorAssembler(inputCols=FEATURES, outputCol="features", handleInvalid="keep")

n_alert = model.transform(assembler.transform(test_pp)).filter(F.col("prediction") == 1).count()
print(f"serving contract check: {n_alert:,} alerts at 0.5  (Phase 4 locked: 1,618 -- must match)")
schema = test_pp.schema                          # the exact JSON contract the producer emits into

serving contract loaded from data/stream/prep (no refit) in 0.0s
serving contract check: 1,618 alerts at 0.5  (Phase 4 locked: 1,618 -- must match)


In [17]:
# ---- Cell 2: streaming job - kafka source -> same assembly -> same model -> alert sinks ----
# One source, two independent checkpoints: alerts parquet (dashboard monitor reads this) and
# the alerts kafka topic (the transport the roadmap promised). startingOffsets=latest keeps a
# blank run clean; on restart the checkpoint's committed offsets win over it, which is the point.
import time as T2
from pyspark.ml.functions import vector_to_array  # if it ever refuses streaming rows: F.udf fallback

BROKER = "kafka:9092"
raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", BROKER)
       .option("subscribe", "transactions")
       .option("startingOffsets", "latest")
       .load())
parsed = raw.select(F.from_json(F.col("value").cast("string"), schema).alias("r")).select("r.*")
stream = model.transform(assembler.transform(parsed))
alerts = (stream.filter(F.col("prediction") == 1)
                .select("TransactionID", "TransactionAmt", "day", "isFraud",
                        vector_to_array("probability")[1].alias("p_fraud")))

q = (alerts.writeStream.format("parquet")
     .option("path", "/home/jovyan/work/data/stream/alerts")
     .option("checkpointLocation", "/home/jovyan/work/data/stream/ckpt_alerts")
     .outputMode("append").trigger(processingTime="2 seconds").start())
qk = (alerts.select(F.to_json(F.struct("TransactionID", "TransactionAmt", "day", "isFraud", "p_fraud"))
                    .alias("value"))
      .writeStream.format("kafka")
      .option("kafka.bootstrap.servers", BROKER)
      .option("topic", "alerts")
      .option("checkpointLocation", "/home/jovyan/work/data/stream/ckpt_alerts_kafka")
      .start())
print("streaming queries live:", q.id, "|", qk.id)

streaming queries live: c1dfbcfb-25ab-4192-89f5-d5e887377df5 | bdfd1aa3-80cc-4420-a3d1-0693cda6b01d


In [18]:
# ---- Cell 3: 60-second scoreboard + alert quality from the parquet sink ----
import pandas as pd
import time as T3

t0, snap = T3.time(), []
while T3.time() - t0 < 60:
    pr = q.lastProgress or {}
    snap.append((round(T3.time() - t0, 1), pr.get("processedRowsPerSecond"), pr.get("numInputRows"),
                 (pr.get("durationMs") or {}).get("addBatch")))
    T3.sleep(5)

sb = pd.DataFrame(snap, columns=["t", "rows/s", "batchIn", "proc ms"])
print(sb.to_string(index=False))
rs, la = sb["rows/s"].dropna(), sb["proc ms"].dropna()
print(f"throughput p50 {rs.median():.0f} | p95 {rs.quantile(0.95):.0f} rows/s")
print(f"processing p50 {la.median():,.0f} | p95 {la.quantile(0.95):,.0f} ms per micro-batch")

ad = spark.read.parquet("/home/jovyan/work/data/stream/alerts")
n    = ad.count()
rate = ad.agg(F.avg(F.col("isFraud").cast("double"))).first()[0]
print(f"stream scoreboard: {n:,} alerts | alert fraud-rate {rate:.0%} (offline was 52%, ~3.9% base rate)")
ad.orderBy(F.desc("p_fraud")).select("TransactionID", "TransactionAmt", "p_fraud", "isFraud").show(5, False)

   t     rows/s  batchIn  proc ms
 0.0        NaN      NaN      NaN
 5.0        NaN      NaN      NaN
10.0        NaN      NaN      NaN
15.0        NaN      NaN      NaN
20.0        NaN      NaN      NaN
25.0        NaN      NaN      NaN
30.0        NaN      NaN      NaN
35.0 460.304165  17585.0  19800.0
40.0 460.304165  17585.0  19800.0
45.0 460.304165  17585.0  19800.0
50.0 243.598459   4300.0  15397.0
55.0 243.598459   4300.0  15397.0
throughput p50 460 | p95 460 rows/s
processing p50 19,800 | p95 19,800 ms per micro-batch
stream scoreboard: 3,406 alerts | alert fraud-rate 53% (offline was 52%, ~3.9% base rate)
+-------------+--------------+------------------+-------+
|TransactionID|TransactionAmt|p_fraud           |isFraud|
+-------------+--------------+------------------+-------+
|3534425      |150.0         |0.9937677032604897|1      |
|3534425      |150.0         |0.9937677032604897|1      |
|3569136      |31.847        |0.9908104342546493|0      |
|3569136      |31.847        |

In [14]:
# ---- Cell 4: stop the streams (the checkpoint dirs stay - they are the resume state) ----
q.stop(); qk.stop()
print("stopped. kill-restart demo, while the producer is still streaming:")
print("  1. re-run Cell 2: same checkpoint dirs -> Spark resumes from committed offsets")
print("  2. re-run Cell 3: alert counts keep rising from where they left, throughput returns")
print("  3. claim on slide: checkpointed offsets = no data loss, no double-count on restart")

stopped. kill-restart demo, while the producer is still streaming:
  1. re-run Cell 2: same checkpoint dirs -> Spark resumes from committed offsets
  2. re-run Cell 3: alert counts keep rising from where they left, throughput returns
  3. claim on slide: checkpointed offsets = no data loss, no double-count on restart
